# McGill COMP551, MINI-PROJECT 3: Odd-One-Out Image Groups
Student names (IDs) - **Kaggle team name**

Mankush Gandhi (261137633) - **Group 88**,
Armain Labeeb (261023603) - **Group 88**, and
Erik Cupsa () - **Group 88**


# Kaggle Competition
Training and test data are available as part of the assignment in the `datasets` folder. To participate in the competition you should upload a CSV file containing your predicted labels for the entire test data to Kaggle.

The test data is divided in half. You receive the correct labels for the first half (in the `datasets` folder), so that you can evaluate your model in this notebook on that part of the test set. You then submit your solution for the entire test set to Kaggle. The final evaluation is based on your predictions for the second half of the test set (private leaderboard). We have step-by-step explanation in the final section of this notebook.


# Report
The purpose of this notebook is for you to provide a concise high level code of your methodology. That is, you need to include all steps necessary to reproduce your results. Please include a high-level explanation of the implementation so that we can understand what you're trying to achieve in each piece of the code. You can import python code here if needed, in order to keep the notebook concise.

**Note:** You need to submit this notebook where you have run all the cells as part of the assignment. **We should not need to rerun your code since this can take a long time.**


## Step 1 — Load Data


In [1]:
import numpy as np

# Mount Google Drive to load the datasets and save outputs.
from google.colab import drive
drive.mount('/content/drive')

import os

# Update PROJECT_DIR here if the folder structure is different.
PROJECT_DIR = '/content/drive/MyDrive/Comp551/mcgill-comp551-winter2026-a3'
DATA_DIR    = os.path.join(PROJECT_DIR, 'datasets')

# Check that all four data files exist before continuing.
print('Checking data files...')
all_ok = True
for fname in ['x_train.npy', 'y_train.npy', 'x_test.npy', 'y_test.npy']:
    path   = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    print(f'  {"✓" if exists else "✗ MISSING"}  {fname}')
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError('One or more data files are missing.')
print('All data files found.')

x = np.load(os.path.join(DATA_DIR, 'x_train.npy'))
y = np.load(os.path.join(DATA_DIR, 'y_train.npy'))

print(f'Training set: {x.shape[0]} groups, each with {x.shape[1]} images of size {x.shape[2]}x{x.shape[3]}')
print(f'Labels: values in {np.unique(y)}')


Mounted at /content/drive
Checking data files...
  ✓  x_train.npy
  ✓  y_train.npy
  ✓  x_test.npy
  ✓  y_test.npy
All data files found.
Training set: 3000 groups, each with 5 images of size 32x32
Labels: values in [0 1 2 3 4]


## Step 2 — PyTorch Model (Under 25,000 Params)


In [2]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Using device: {device}')

x_train_split, x_val_split, y_train_split, y_val_split = train_test_split(
    x, y, test_size=0.15, random_state=42, stratify=y
)

if x_train_split.max() > 1.0:
    x_train_split = x_train_split.astype(np.float32) / 255.0
    x_val_split = x_val_split.astype(np.float32) / 255.0

class OddOneOutDataset(Dataset):
    def __init__(self, images, labels=None, transform=None):
        images = torch.as_tensor(images, dtype=torch.float32)
        if images.ndim == 4:
            images = images.unsqueeze(2)  # (N, 5, 1, H, W)
        self.images = images.contiguous()
        self.labels = None if labels is None else torch.as_tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return self.images.shape[0]

    def __getitem__(self, idx):
        group = self.images[idx]
        if self.transform is not None:
            group = torch.stack([self.transform(img) for img in group], dim=0)
        if self.labels is None:
            return group
        return group, self.labels[idx]

train_transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=20,
        translate=(0.12, 0.12),
        scale=(0.85, 1.15),
        shear=10,
        interpolation=InterpolationMode.BILINEAR,
        fill=0,
    ),
    transforms.RandomHorizontalFlip(p=0.5),
])

train_dataset = OddOneOutDataset(x_train_split, y_train_split, transform=train_transform)
val_dataset = OddOneOutDataset(x_val_split, y_val_split, transform=None)

num_workers = 0  # Avoid multiprocessing pickling issues in notebook execution
pin_memory = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=(num_workers > 0),
)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


Using device: cpu
Train batches: 40 | Val batches: 4


In [3]:
class DSConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size=3, stride=stride, padding=1, groups=in_ch, bias=False),
            nn.BatchNorm2d(in_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class OddOneOutCosineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 24, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(24),
            nn.SiLU(inplace=True),
            DSConvBlock(24, 24, stride=1),
            DSConvBlock(24, 48, stride=2),
            DSConvBlock(48, 64, stride=1),
            DSConvBlock(64, 80, stride=2),
            DSConvBlock(80, 80, stride=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
        self.log_temperature = nn.Parameter(torch.tensor(np.log(0.07), dtype=torch.float32))

    def forward(self, x):
        b, k, c, h, w = x.shape
        z = self.encoder(x.view(b * k, c, h, w))
        z = F.normalize(z, p=2, dim=-1)
        z = z.view(b, k, -1)

        sum_z = z.sum(dim=1, keepdim=True)
        loo_mean = (sum_z - z) / (k - 1)
        sim = F.cosine_similarity(z, loo_mean, dim=-1, eps=1e-8)

        temperature = torch.exp(self.log_temperature).clamp(min=1e-3, max=10.0)
        logits = -sim / temperature
        return logits

model = OddOneOutCosineNet().to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {total_params:,}')
assert total_params <= 25_000, f'Model too large! {total_params:,} > 25,000'


Trainable params: 19,817


In [4]:
epochs = 90
max_lr = 3e-3

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=max_lr, weight_decay=5e-3)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    epochs=epochs,
    steps_per_epoch=len(train_loader),
    pct_start=0.15,
    anneal_strategy='cos',
    div_factor=20.0,
    final_div_factor=500.0,
)

best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())
non_blocking = device.type == 'cuda'

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=non_blocking)
        yb = yb.to(device, non_blocking=non_blocking)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item() * yb.size(0)
        train_correct += (logits.argmax(dim=1) == yb).sum().item()
        train_total += yb.size(0)

    train_loss /= train_total
    train_acc = 100.0 * train_correct / train_total

    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=non_blocking)
            yb = yb.to(device, non_blocking=non_blocking)
            logits = model(xb)
            val_correct += (logits.argmax(dim=1) == yb).sum().item()
            val_total += yb.size(0)

    val_acc = 100.0 * val_correct / val_total

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())

    lr_now = optimizer.param_groups[0]['lr']
    temp_now = torch.exp(model.log_temperature).item()
    print(
        f'Epoch {epoch:03d}/{epochs} | '
        f'loss {train_loss:.4f} | train {train_acc:.2f}% | val {val_acc:.2f}% | '
        f'lr {lr_now:.2e} | temp {temp_now:.4f}'
    )

model.load_state_dict(best_state)
print(f'Best validation accuracy: {best_val_acc:.2f}%')


Epoch 001/90 | loss 1.6824 | train 22.51% | val 25.78% | lr 1.89e-04 | temp 0.0704
Epoch 002/90 | loss 1.6231 | train 24.63% | val 22.00% | lr 3.02e-04 | temp 0.0709
Epoch 003/90 | loss 1.6094 | train 24.94% | val 22.44% | lr 4.85e-04 | temp 0.0715
Epoch 004/90 | loss 1.5975 | train 27.45% | val 25.56% | lr 7.26e-04 | temp 0.0722
Epoch 005/90 | loss 1.5899 | train 28.04% | val 28.44% | lr 1.01e-03 | temp 0.0734
Epoch 006/90 | loss 1.5762 | train 28.27% | val 26.00% | lr 1.33e-03 | temp 0.0749
Epoch 007/90 | loss 1.5787 | train 28.94% | val 28.67% | lr 1.66e-03 | temp 0.0770
Epoch 008/90 | loss 1.5753 | train 29.92% | val 30.67% | lr 1.99e-03 | temp 0.0800
Epoch 009/90 | loss 1.5656 | train 30.55% | val 29.33% | lr 2.29e-03 | temp 0.0828
Epoch 010/90 | loss 1.5521 | train 31.80% | val 32.89% | lr 2.56e-03 | temp 0.0854
Epoch 011/90 | loss 1.4803 | train 38.16% | val 38.22% | lr 2.77e-03 | temp 0.0845
Epoch 012/90 | loss 1.1722 | train 59.80% | val 60.67% | lr 2.92e-03 | temp 0.0834
Epoc

## Step 3 — Evaluate and Export Kaggle CSV


In [6]:
from sklearn.metrics import accuracy_score

x_test = np.load(os.path.join(DATA_DIR, 'x_test.npy'))
y_test_half = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

if x_test.max() > 1.0:
    x_test = x_test.astype(np.float32) / 255.0

x_test_tensor = torch.tensor(x_test, dtype=torch.float32).unsqueeze(2)
test_dataset = torch.utils.data.TensorDataset(x_test_tensor)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

model.eval()
all_predictions = []
with torch.no_grad():
    for (inputs,) in test_loader:
        outputs = model(inputs.to(device))
        predicted = torch.argmax(outputs, dim=1)
        all_predictions.extend(predicted.cpu().tolist())

all_predictions = np.array(all_predictions)
public_accuracy = accuracy_score(y_test_half, all_predictions[:len(y_test_half)])
print(f'Public test accuracy: {public_accuracy * 100:.2f}%')


Public test accuracy: 66.50%


In [10]:
import pandas as pd

def generate_csv_kaggle(predictions, filename='predicted_labels.csv'):
    """
    Save predictions in the format Kaggle expects:
      Id       — row index as a string (0, 1, 2, ...)
      Category — predicted outlier position (0–4) as a string

    We predict on all 2000 test groups. The first 1000 score on the
    public leaderboard; the last 1000 score on the private leaderboard
    (revealed after the deadline).
    """
    df = pd.DataFrame({
        'Id':       np.arange(len(predictions)).astype(str),
        'Category': predictions.astype(str)
    })
    df.to_csv(filename, index=False)
    print(f'Saved {len(predictions)} predictions to {filename}')
    print('First 5 rows:')
    print(df.head())


csv_path = os.path.join(PROJECT_DIR, 'predicted_labels.csv')
generate_csv_kaggle(all_predictions, csv_path)

# Direct download from Colab.
from google.colab import files
files.download(csv_path)


Saved 2000 predictions to /content/drive/MyDrive/Comp551/mcgill-comp551-winter2026-a3/predicted_labels.csv
First 5 rows:
  Id Category
0  0        3
1  1        3
2  2        4
3  3        0
4  4        0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>